# Spam Email Detection – Exploratory Analysis

This notebook follows the same steps as `train_model.py`, with explanations. Run `python download_data.py` first.

> Note: nothing here is saved to disk. Use `python train_model.py` to create the model used by the app.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable (this notebook lives in notebooks/)
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

from spam_utils import load_dataset

## 1. Load and clean the data
`load_dataset` removes missing values and duplicates and standardises labels to `spam` / `ham`.

In [ ]:
data = load_dataset()
print(data.shape)
data.head()

## 2. Explore the data

In [ ]:
print(data['label'].value_counts())
sns.countplot(data=data, x='label', order=['ham', 'spam'])
plt.title('Ham vs spam messages')
plt.show()

In [ ]:
# Do spam messages tend to be longer?
data['length'] = data['message'].str.len()
print(data.groupby('label')['length'].describe())

## 3. Split into train and test sets
`stratify` keeps the same spam/ham ratio in both sets. `random_state` makes the split repeatable.

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    data['message'], data['label'], test_size=0.2, random_state=42, stratify=data['label'])
print(len(x_train), 'training messages,', len(x_test), 'test messages')

## 4. Build and train the pipeline
The pipeline fits TF-IDF on training data only, which prevents data leakage.

In [ ]:
model = Pipeline([
    ('tfidf', TfidfVectorizer(lowercase=True, stop_words='english')),
    ('classifier', MultinomialNB()),
])
model.fit(x_train, y_train)

## 5. Evaluate on the test set

In [ ]:
predictions = model.predict(x_test)
print('Accuracy :', round(accuracy_score(y_test, predictions), 4))
print('Precision:', round(precision_score(y_test, predictions, pos_label='spam'), 4))
print('Recall   :', round(recall_score(y_test, predictions, pos_label='spam'), 4))
print('F1-score :', round(f1_score(y_test, predictions, pos_label='spam'), 4))

matrix = confusion_matrix(y_test, predictions, labels=['ham', 'spam'])
sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=['ham', 'spam'], yticklabels=['ham', 'spam'])
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.show()

## 6. Which words push a message towards spam?
A higher log-probability difference means the word is more typical of spam.

In [ ]:
import numpy as np
tfidf = model.named_steps['tfidf']
classifier = model.named_steps['classifier']
words = np.array(tfidf.get_feature_names_out())
spam_index = list(classifier.classes_).index('spam')
ham_index = list(classifier.classes_).index('ham')
difference = classifier.feature_log_prob_[spam_index] - classifier.feature_log_prob_[ham_index]
print('Top spam-like words:', list(words[np.argsort(difference)[-15:][::-1]]))

## 7. Try your own messages

In [ ]:
for text in ['Congratulations! You won a free prize, call now',
             'Are we still meeting for lunch tomorrow?']:
    print(model.predict([text])[0], '<-', text)